In [ ]:
def parse_script(script):
    """
    解析AD9361初始化脚本，提取命令和参数
    """
    commands = []
    lines = script.strip().split('\n')
    
    for line in lines:
        line = line.strip()

        # 跳过空行和不需要处理的命令
        if not line or line.startswith('RESET') or line.startswith('ReadPartNumber') or line.startswith('REFCLK_Scale'):
            continue


        ori_comment = ''
        
        if '//' in line:
            line, ori_comment = line.split('//', 1)
            ori_comment = ori_comment.strip()
            ori_comment = f", {ori_comment}"

            # print(f"Found comment: {line}")
            if not line:
                # print(f"Only comment line: {ori_comment}")
                commands.append(('COMMENT', ori_comment))

        # 处理SPIWrite命令
        if line.startswith('SPIWrite'):
            parts = line.split()

            if len(parts) >= 2:
                addr_data = parts[1].split(',')
                if len(addr_data) == 2:
                    addr_hex = addr_data[0].strip()
                    data_hex = addr_data[1].strip()
                    # 确保地址和数据是16进制格式
                    if all(c in '0123456789ABCDEF' for c in addr_hex) and all(c in '0123456789ABCDEF' for c in data_hex):
                        addr_int = int(addr_hex, 16)
                        data_int = int(data_hex, 16)
                        commands.append((('SPIWrite', addr_int, data_int), ori_comment))

        elif line.startswith('WAIT_CALDONE'):
            # print("WARNING: WAIT_CALDONE converted to WAIT 20ms")
            commands.append((('WAIT', 20), ", " + line + ori_comment))

        # 处理WAIT命令
        elif line.startswith('WAIT'):
            parts = line.split()
            if len(parts) >= 2:
                try:
                    wait_time = int(parts[1])
                    commands.append((('WAIT', wait_time), ori_comment))
                except ValueError:
                    pass

        # elif line.startswith('SPIRead'):
        #     commands.append((('WAIT', 20), ori_comment))
    commands.append((('SPIWrite', 0x014, 0x07), ", " + "State Machine Set"))
    commands.append((('SPIWrite', 0x014, 0x23), ", " + "State Machine Set"))

    return commands

def generate_rom_code(commands):
    """
    生成ROM代码
    """
    rom_entries = []
    addr = 0
    
    for cmd_full in commands:
        cmd, ori_comment = cmd_full
        if cmd[0] == 'SPIWrite':
            addr_hex = f"{cmd[1]:03X}"
            data_hex = f"{cmd[2]:02X}"
            rom_entries.append(f"12'd{addr:04}: data <= 20'h{addr_hex}{data_hex}; // SPIWrite [{cmd[1]:03X}]={cmd[2]:02X}{ori_comment}")
            addr += 1
        elif cmd[0] == 'WAIT':
            # WAIT命令编码为0x1 + 时间(ms) + 填充
            wait_time = min(cmd[1], 0xFFFF)  # 限制最大等待时间
            rom_entries.append(f"12'd{addr:04}: data <= 20'h40000; // WAIT {cmd[1]} ms{ori_comment}")
            addr += 1
        elif cmd == 'COMMENT':
            # print(f"Adding comment line: {ori_comment}")
            rom_entries.append("// "+ori_comment[2:])
    
    # 添加结束标记
    rom_entries.append(f"12'd{addr}: data <= 20'h80000; // INIT_END")
    
    return rom_entries

def main():
    # 读取输入脚本
    with open(r'C:/Users/DDDD/Desktop/ad9361 new', 'r') as f:
        script = f.read()
    
    # 解析脚本
    commands = parse_script(script)
    
    # 生成ROM代码
    rom_code = generate_rom_code(commands)
    
    # 输出ROM代码
    print("module ad9361_cfg_rom(")
    print("    input              clk,")
    print("    input      [11:0]  addr,")
    print("    output reg [19:0]  data")
    print(");")
    print("")
    print("    always @(posedge clk) begin")
    print("        case(addr)")
    
    for entry in rom_code:
        print(f"            {entry}")
    
    print("            default:")
    print("                data <= 20'h00000;")
    print("        endcase")
    print("    end")
    print("")
    print("endmodule")

if __name__ == "__main__":
    main()

module ad9361_cfg_rom(
    input              clk,
    input      [11:0]  addr,
    output reg [19:0]  data
);

    always @(posedge clk) begin
        case(addr)
            // ************************************************************
            // AD9361 R2 Auto Generated Initialization Script:  This script was
            // generated using the AD9361 Customer software Version 2.1.1
            // ************************************************************
            // Profile: Custom
            // REFCLK_IN: 40.000 MHz
            12'd0000: data <= 20'h3DF01; // SPIWrite [3DF]=01
            12'd0001: data <= 20'h29514; // SPIWrite [295]=14, Power up XO path (Default)
            12'd0002: data <= 20'h2A60E; // SPIWrite [2A6]=0E, Enable Master Bias
            12'd0003: data <= 20'h2A80E; // SPIWrite [2A8]=0E, Set Bandgap Trim
            12'd0004: data <= 20'h29208; // SPIWrite [292]=08, Set DCXO Coarse Tune[5:0]
            12'd0005: data <= 20'h29380; // SPIWrite [293]=8